# SQL Dataset Merge + Cleaning + Preprocessing

Bu notebook, iki farkli SQL dataset'ini ortak formata getirir, birlestirir,
`cleaned_sql_data.csv` mantigina benzer sekilde temizler ve preprocessing uygular.
Sonunda yeni bir CSV dosyasi olusturur.

In [3]:
import os
import re
from urllib.parse import unquote_plus

import pandas as pd

## 1) Veri Kaynaklarini Yukle, Karistir ve Birlestir

- `looking for beter dataset/dataset.csv` dosyasindan sadece `full_query` ve `label` alinir.
- Kolon adlari `Sentence` ve `Label` olarak degistirilir.
- Ilk dataset tamamen rastgele karistirilir ve indeks sifirlanir (`random_state=42`).
- `SQLiV3.csv` dosyasi da `Sentence` ve `Label` formatina getirilir.
- Iki dataset birlestirilir.
- Birlestirilen yeni dataset son kez tamamen rastgele karistirilir ve indeks yeniden sifirlanir (`random_state=42`).

In [4]:
PATH_BETTER = "./data/looking for beter dataset/dataset.csv"  # Ilk dataset dosya yolu
PATH_SQLIV3 = "./data/SQLiV3.csv"  # Ikinci dataset dosya yolu
OUTPUT_PATH = "./data/merged_cleaned_preprocessed.csv"  # Cikti dosya yolu

# 1) Ilk dataseti oku
better_df = pd.read_csv(PATH_BETTER, low_memory=False)  # Tip karmasasi uyarisini azaltarak oku
better_df = better_df[["full_query", "label"]].copy()  # Sadece istenen kolonlari al
better_df = better_df.rename(columns={"full_query": "Sentence", "label": "Label"})  # Kolon isimlerini standardize et
better_df = better_df.sample(frac=1, random_state=42).reset_index(drop=True)  # Tam rastgele karistir + indeks sifirla

# 2) SQLiV3 dataseti oku ve kolonlarini hazirla
sqliv3_df = pd.read_csv(PATH_SQLIV3)  # SQLiV3 dosyasini yukle
sqliv3_df = sqliv3_df.loc[:, ~sqliv3_df.columns.str.contains(r"^Unnamed", case=False)]  # Unnamed kolonlari temizle

# 3) Gerekli kolonlari kontrol et
if "Sentence" not in sqliv3_df.columns or "Label" not in sqliv3_df.columns:
    raise ValueError("SQLiV3.csv dosyasinda 'Sentence' ve 'Label' kolonlari bulunamadi.")

# 4) SQLiV3 tarafindan sadece hedef kolonlari al
sqliv3_df = sqliv3_df[["Sentence", "Label"]].copy()  # Ortak kolon seti

# 5) Iki dataseti birlestir
merged_df = pd.concat([sqliv3_df, better_df], ignore_index=True)  # Alt alta birlestir

# 6) Birlestirilmis buyuk dataseti son kez rastgele karistir
merged_df = merged_df.sample(frac=1, random_state=42).reset_index(drop=True)  # Final shuffle + reset_index

# 7) Boyut bilgilerini yazdir
print("SQLiV3 shape:", sqliv3_df.shape)  # SQLiV3 satir/kolon sayisi
print("Better dataset (first shuffle) shape:", better_df.shape)  # Ilk karistirilmis dataset boyutu
print("Merged dataset (final shuffle) shape:", merged_df.shape)  # Final karistirilmis birlesik boyut
merged_df.head()  # Ornek satirlar

C:\Users\Mrt\AppData\Local\Temp\ipykernel_13504\4262484001.py:6: DtypeWarning: Columns (0: attack_stage, 1: tamper_method, 2: attack_status, 3: attack_id, 4: attack_technique) have mixed types. Specify dtype option on import or set low_memory=False.
  better_df = pd.read_csv(PATH_BETTER)


SQLiV3 shape: (30919, 2)
Better dataset shape: (3688977, 2)
Merged shape: (3719896, 2)


,Sentence,Label
0,""" or pg_sleep ( __TIME__ ) --",1
1,create user name identified by pass123 tempora...,NaN
2,AND 1 = utl_inaddr.get_host_address ( ...,1
3,select * from users where id = '1' or @ @1 ...,1
4,"select * from users where id = 1 or 1#"" ( ...",1


## 2) Data Cleaning + Preprocessing

Bu adimda:
- NaN ve bos satirlar temizlenir
- URL decoding uygulanir
- Metinler kucuk harfe cevrilir
- Gereksiz bosluklar normalize edilir
- Label degerleri binary (0/1) formata donusturulur
- Duplicate satirlar temizlenir

In [5]:
def preprocess_sentence(text):
    text = unquote_plus(str(text))
    text = text.lower().strip()
    text = re.sub(r"\s+", " ", text)
    return text


def to_binary_label(value):
    value = str(value).strip().lower()
    malicious_labels = {"1", "true", "yes", "sqli", "malicious", "attack"}
    return 1 if value in malicious_labels else 0


clean_df = merged_df.copy()

# Null ve bos veri temizligi
clean_df = clean_df.dropna(subset=["Sentence", "Label"]).copy()
clean_df["Sentence"] = clean_df["Sentence"].astype(str)
clean_df["Label"] = clean_df["Label"].astype(str)
clean_df = clean_df[clean_df["Sentence"].str.strip() != ""].copy()

# Preprocessing
clean_df["Sentence"] = clean_df["Sentence"].apply(preprocess_sentence)
clean_df["Label"] = clean_df["Label"].apply(to_binary_label)

# Duplicate temizligi (Sentence + Label bazli)
clean_df = clean_df.drop_duplicates(subset=["Sentence", "Label"]).reset_index(drop=True)

print("Cleaned + preprocessed shape:", clean_df.shape)
print(clean_df["Label"].value_counts(dropna=False))
clean_df.head()

Cleaned + preprocessed shape: (2322915, 2)
Label
0    2001191
1     321724
Name: count, dtype: int64


,Sentence,Label
0,""" or pg_sleep ( __time__ ) --",1
1,and 1 = utl_inaddr.get_host_address ( ( select...,1
2,select * from users where id = '1' or @ @1 = 1...,1
3,"select * from users where id = 1 or 1#"" ( unio...",1
4,select name from syscolumns where id = ( selec...,1


## 3) Yeni CSV Dosyasini Kaydet

In [ ]:
# Cikti klasorunu garanti et
os.makedirs(os.path.dirname(OUTPUT_PATH), exist_ok=True)

# Sadece istenen kolonlarla kaydet
final_df = clean_df[["Sentence", "Label"]].copy()
final_df.to_csv(OUTPUT_PATH, index=False, encoding="utf-8")

print("Yeni CSV olusturuldu:")
print(OUTPUT_PATH)
print("Satir sayisi:", len(final_df))
final_df.head(5)

Yeni CSV olusturuldu:
./data/merged_cleaned_preprocessed.csv
Satir sayisi: 2322915


,Sentence,Label
0,""" or pg_sleep ( __time__ ) --",1
1,and 1 = utl_inaddr.get_host_address ( ( select...,1
2,select * from users where id = '1' or @ @1 = 1...,1
3,"select * from users where id = 1 or 1#"" ( unio...",1
4,select name from syscolumns where id = ( selec...,1


In [7]:
final_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 2322915 entries, 0 to 2322914
Data columns (total 2 columns):
 #   Column    Dtype
---  ------    -----
 0   Sentence  str  
 1   Label     int64
dtypes: int64(1), str(1)
memory usage: 35.4 MB
